# 02 Compute Types

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Know every kind of Databricks compute (serverless notebooks and jobs, all-purpose clusters, job clusters, SQL warehouses, instance pools), when to use each, what you configure on classic clusters (runtime, node types, autoscaling, access mode, policies), and how cost relates to each choice.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Choosing compute is a daily cost and reliability decision: interactive work on one kind, production jobs on another, BI on a third. The exam's platform domain asks which compute fits a scenario, and interviewers ask "why not run production jobs on an all-purpose cluster?". Free Edition gives you serverless only, so this lesson also explains the classic options you'll meet in company workspaces.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The compute families</b><br>
| Compute | Used for | Lifecycle | Configuration | Billing idea |<br>|---|---|---|---|---|<br>| <b>Serverless notebooks</b> | Interactive notebooks | Starts in seconds, managed | Environment version, a few Spark settings | Per use |<br>| <b>Serverless jobs</b> | Scheduled jobs and pipelines | Per run, managed | Performance vs cost mode | Per use |<br>| <b>All-purpose (classic) cluster</b> | Interactive development, shared by users | Starts in minutes, runs until terminated or auto-terminated | Full control | While running |<br>| <b>Job (classic) cluster</b> | One job run | Created for the run, terminated after | Full control, defined in the job | While running, cheaper rate than all-purpose |<br>| <b>SQL warehouse</b> (serverless, pro, classic) | SQL editor, dashboards, BI tools | Starts on query, auto-stops | Size (T-shirt), scaling, auto-stop | While running |<br>| <b>Instance pool</b> | Faster classic cluster start | Idle VMs kept warm | Min/max idle instances | Cloud VM cost of idle instances |
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A company ran its nightly ETL on a shared all-purpose cluster that stayed up 24/7 "so it's ready". Moving the jobs to job clusters (later serverless jobs) that start for each run and stop afterwards cut that compute bill by 60%, and an analyst's heavy notebook could no longer slow down production loads. Analysts moved their BI dashboards to a serverless SQL warehouse with a 10-minute auto-stop.
</div>

## Classic cluster settings you should recognize

| Setting | Meaning | Typical choice |
|---|---|---|
| Databricks Runtime (DBR) | Spark version plus libraries. LTS versions get long-term support | Latest LTS for production |
| Photon | Native vectorized engine | On for SQL and DataFrame-heavy work |
| Node types | Driver and worker VM sizes (memory, CPU, storage optimized) | Match the workload |
| Autoscaling | Min and max workers | On for variable loads |
| Auto termination | Stop after N idle minutes | Always set, for example 30–60 minutes |
| Access mode | **Standard** (shared, Unity Catalog, multi-user) or **Dedicated** (single user/group) | Standard for most. Dedicated for some ML and RDD workloads |
| Cluster policies | Admin-defined limits and defaults | Enforce cost and security rules |
| Init scripts, libraries | Install software at start | Prefer environments, or libraries on job clusters |
| Spark config | Low-level settings | Only when needed |

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Free Edition doesn't let you create classic clusters, pools or policies. The cells below inspect what you <b>can</b> see: the serverless environment, the Spark version and the SQL warehouses.
</div>

## 1. Inspect your current compute

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Prints the Spark version, Python version and a few settings of the compute this notebook runs on, and checks whether it is serverless (Spark Connect).<br><br>
<b>Why it matters</b><br>When debugging or reporting an issue, the first facts anyone asks for are the runtime and compute type.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The Spark and Python versions, the session type (Spark Connect on serverless), and the shuffle partition setting.
</div>

In [0]:
import platform, sys

print("Spark version  :", spark.version)
print("Python version :", platform.python_version())
print("Session class  :", type(spark).__module__)        # pyspark.sql.connect... means Spark Connect (serverless)
for key in ["spark.sql.shuffle.partitions", "spark.sql.ansi.enabled", "spark.sql.session.timeZone"]:
    try:
        print(f"{key:<32}", spark.conf.get(key))
    except Exception:
        print(f"{key:<32} not readable")

## 2. The serverless environment

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists a few preinstalled Python libraries and their versions, and shows how to add a library for this notebook.<br><br>
<b>Why it matters</b><br>On serverless, the <b>environment version</b> (set in the notebook's Environment side panel) decides which Python and library versions you get. You add libraries per notebook with <code>%pip install</code> or in the Environment panel, instead of installing them on a cluster.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Versions of pandas, pyarrow, numpy and the Databricks SDK.
</div>

In [0]:
from importlib.metadata import version, PackageNotFoundError

for pkg in ["pandas", "pyarrow", "numpy", "databricks-sdk"]:
    try:
        print(f"{pkg:<16} {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:<16} not installed")

```python
# Add a library to this notebook's serverless environment (restarts the Python process):
%pip install great-expectations==1.*
```

## 3. Look at the compute available in the workspace

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses the Databricks SDK (preinstalled, authenticated automatically in notebooks) to list SQL warehouses and clusters.<br><br>
<b>Why it matters</b><br>The SDK is how automation scripts create, inspect and manage compute. In Free Edition you'll see a serverless SQL warehouse and no classic clusters.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
The SDK call works in Databricks notebooks without extra authentication. If your workspace restricts it, the cell prints a note.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One or more SQL warehouses with their size, state and type, and an empty (or restricted) cluster list.
</div>

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
try:
    for wh in w.warehouses.list():
        print(f"warehouse: {wh.name:<30} size={wh.cluster_size} state={wh.state} serverless={wh.enable_serverless_compute}")
except Exception as e:
    print("warehouses not listable:", type(e).__name__)
try:
    clusters = list(w.clusters.list())
    print("classic clusters:", [c.cluster_name for c in clusters] or "none")
except Exception as e:
    print("clusters not listable:", type(e).__name__)

## 4. Choosing compute: decision guide

```
Is it SQL only (queries, dashboards, BI tools)?
   └─ yes ─▶ SQL warehouse (serverless if available)
Is it a scheduled / automated pipeline?
   └─ yes ─▶ Serverless jobs compute (or a job cluster on classic)
             Lakeflow Declarative Pipelines: serverless pipeline compute
Is it interactive development or exploration?
   └─ yes ─▶ Serverless notebook (or a shared all-purpose cluster, standard access mode)
Do you need GPUs, special libraries, init scripts, RDD APIs or Scala/R?
   └─ yes ─▶ Classic cluster (dedicated access mode where required)
```

| Scenario | Best choice | Why |
|---|---|---|
| Nightly ETL job | Serverless jobs / job cluster | Pays only during the run, isolated from interactive users |
| Analyst dashboards | Serverless SQL warehouse | Fast start, auto-stop, BI concurrency |
| Ad hoc notebook exploration | Serverless notebook | No setup, starts in seconds |
| Deep learning training | Classic GPU cluster | Hardware choice |
| Legacy RDD / Scala code | Classic cluster, dedicated access | Serverless and standard access restrict these APIs |

## 5. Cost habits

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the habits that keep compute costs down, and runs a tiny example of measuring work instead of guessing.<br><br>
<b>Why it matters</b><br>Compute is billed in <b>DBUs</b> (Databricks Units) per hour plus, on classic compute, the cloud VMs. Most waste comes from idle clusters, oversized compute and inefficient queries. <code>05_optimization/07_cost_awareness</code> goes deeper.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Cost habits</b>
<ul><li>Always set <b>auto termination</b> / auto-stop</li><li>Use <b>job compute</b> (or serverless) for production, not all-purpose clusters</li><li>Right-size: start small, scale up when the query profile shows it's needed</li><li>Make queries efficient first (filter early, avoid UDFs, fix skew). Faster queries cost less</li><li>Use cluster policies to cap sizes, and tags to attribute costs to teams</li></ul>
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The time taken by the same aggregation written efficiently and inefficiently (a Python UDF), as a reminder that code choices drive compute cost.
</div>

In [0]:
import time
from pyspark.sql import functions as F
from pyspark.sql.functions import udf

data = spark.range(2_000_000).withColumn("x", F.col("id") % 1000)

@udf("long")
def plus_one(v):
    return v + 1

for label, col in [("built-in expression", F.col("x") + 1), ("Python UDF", plus_one("x"))]:
    start = time.time()
    data.select(F.sum(col)).collect()
    print(f"{label:<20} {time.time() - start:.2f}s")

## Under the hood

```
Classic compute                               Serverless compute
your cloud account:                           Databricks-managed account:
  driver VM + worker VMs (you pay the cloud)    pool of warm machines, shared securely
  start: provision VMs (minutes)                start: attach to warm capacity (seconds)
  you choose runtime, nodes, scaling            Databricks chooses and scales
  DBUs + VM cost while running                  DBUs only (include infrastructure), per use
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata only</span> choosing compute doesn't change results, only cost, speed and isolation.

**Access modes and Unity Catalog:** standard (shared) access mode runs each user's code in isolation so one cluster can serve many users with Unity Catalog permissions. Dedicated access mode gives one user or group full access, needed for some workloads such as certain ML libraries or RDD APIs.

In [0]:
print("Spark Connect session (serverless-style):", "connect" in type(spark).__module__)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: production jobs on an all-purpose cluster</b><br>
Higher DBU rate, always-on cost, and interference from interactive users. Use job compute or serverless jobs.<br><br>
<b>⛔ Problem: cluster left running all weekend</b><br>No auto termination. Always set it, and use policies to enforce it.<br><br>
<b>⛔ Problem: code works on a classic cluster but not on serverless</b><br>It uses <code>sparkContext</code>, RDDs, caching or unsupported configs. Rewrite with DataFrame APIs (see <code>00_03</code>).<br><br>
<b>⛔ Problem: SQL dashboards are slow on a notebook cluster</b><br>Use a SQL warehouse, which is built for BI concurrency and caching.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What types of compute does Databricks offer?</b><br>
Serverless compute for notebooks, jobs and pipelines, classic all-purpose clusters for interactive work, classic job clusters created per job run, SQL warehouses for SQL and BI, and instance pools that keep idle VMs warm to speed up classic cluster starts.<br><br>

<b>🎤 2. Why use job clusters or serverless jobs instead of an all-purpose cluster for production?</b><br>
They exist only for the run, so you don't pay for idle time and the DBU rate is lower. Each run gets a clean, isolated environment that interactive users can't disturb.<br><br>

<b>🎤 3. What is a SQL warehouse?</b><br>
Compute optimized for SQL queries, dashboards and BI tools, sized in T-shirt sizes with auto-stop and scaling for concurrency. Serverless warehouses start in seconds.<br><br>

<b>🎤 4. What are the trade-offs of serverless compute?</b><br>
You get fast start-up, no infrastructure management and usage-based billing. In exchange you give up control over runtime details, node types, init scripts and some APIs, such as RDDs, <code>sparkContext</code> and caching.<br><br>

<b>🎤 5. What is a cluster policy?</b><br>
An admin-defined set of rules and defaults for cluster creation, such as allowed node types, maximum workers, required tags and auto termination. It controls cost and enforces standards.<br><br>

<b>🎤 6. What is the difference between standard and dedicated access mode?</b><br>
Standard (shared) access mode isolates each user's code so many users can share a Unity Catalog-enabled cluster safely. Dedicated access mode assigns the cluster to one user or group, which supports workloads that need full access.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A team runs a nightly batch job and wants to minimize cost while isolating it from interactive users. Which compute should they use?</b><br>
A. A shared all-purpose cluster that runs 24/7<br>
B. Job compute (or serverless jobs compute) created for each run<br>
C. A SQL warehouse<br>
D. An instance pool alone<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which compute is designed for running SQL queries and dashboards from BI tools?</b><br>
A. All-purpose cluster<br>
B. SQL warehouse<br>
C. Job cluster<br>
D. Instance pool<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q3. What does an instance pool provide?</b><br>
A. Automatic SQL query caching<br>
B. A set of idle, ready VMs that reduce classic cluster start and autoscaling time<br>
C. Serverless compute for notebooks<br>
D. Data storage for Delta tables<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Record your compute facts: Spark version, Python version, session type and environment libraries</li><li>Using the SDK, find your SQL warehouse's size and auto-stop setting (<code>auto_stop_mins</code>)</li><li>For each scenario, choose the compute and write one reason: (a) hourly streaming ingestion, (b) a data scientist training a GPU model, (c) a finance dashboard used by 50 people, (d) a one-off data exploration</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Compute facts
print(spark.version, platform.python_version(), type(spark).__module__)

# 2. Warehouse size and auto-stop
for wh in WorkspaceClient().warehouses.list():
    print(wh.name, wh.cluster_size, "auto-stop:", wh.auto_stop_mins, "min")

# 3. (a) serverless jobs or a job cluster: scheduled, isolated, pay per run
#    (b) a classic GPU cluster (dedicated access): needs specific hardware and libraries
#    (c) a serverless SQL warehouse: BI concurrency, fast start, auto-stop
#    (d) a serverless notebook: no setup, starts in seconds

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Serverless (notebooks, jobs, pipelines, SQL) for most work: fast start, managed, pay per use</li><li>Classic all-purpose clusters for interactive work needing full control. Job clusters for production runs</li><li>SQL warehouses for SQL, dashboards and BI tools</li><li>Instance pools speed up classic cluster starts</li><li>Classic settings: runtime (LTS), Photon, node types, autoscaling, auto termination, access mode, policies</li><li>Production on job compute or serverless jobs, never on always-on all-purpose clusters</li></ul>
</div>

| Need | Compute |
|---|---|
| Interactive notebook | Serverless notebook / all-purpose cluster |
| Scheduled pipeline | Serverless jobs / job cluster |
| SQL and BI | SQL warehouse |
| GPUs, special libs, RDDs | Classic cluster (dedicated) |
| Faster classic starts | Instance pool |

## Git commit

```
git add 04_databricks/02_compute_types.ipynb
git commit -m "add 04_02 compute types notebook"
```